In [3]:
# ── Standard library ───────────────────────────────────────────────────────
import os
import re
from pathlib import Path
from datetime import datetime

# ── Data science ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Geospatial ─────────────────────────────────────────────────────────────
from geopy.distance import geodesic

# ── Display settings ────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
plt.style.use("dark_background")

# ──────────────────────────────────────────────────────────────────────────
# PROJECT ROOT — pathlib makes paths OS-agnostic (works on Windows AND Linux)
# Path(__file__) would be the script path, but in notebooks we use Path.cwd()
# ──────────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("C:/xcas-ga-comms-assistant")  # Change if different

# ── Data paths ──────────────────────────────────────────────────────────────
ADSB_PATH    = PROJECT_ROOT / "tartan_data/kbtp/raw/2020/10-22-20/1.csv"
AUDIO_DIR    = PROJECT_ROOT / "tartan_data/kbtp/2020/10/10-22-20_audio"
WEATHER_PATH = PROJECT_ROOT / "tartan_data/weather/BTP.csv"

# ── Output paths ────────────────────────────────────────────────────────────
INTERIM_DIR  = PROJECT_ROOT / "data/interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# ── Airport anchor (KBTP — Butler County Regional Airport, PA) ──────────────
KBTP_LAT  = 40.7769
KBTP_LON  = -79.9697
KBTP_ELEV = 1248   # feet MSL
KBTP_NAME = "Butler"   # Used in radio callouts: "Butler Traffic"

print("✅ Paths configured")
print(f"   ADS-B  : {ADSB_PATH}")
print(f"   Audio  : {AUDIO_DIR}")
print(f"   Weather: {WEATHER_PATH}")

✅ Paths configured
   ADS-B  : C:\xcas-ga-comms-assistant\tartan_data\kbtp\raw\2020\10-22-20\1.csv
   Audio  : C:\xcas-ga-comms-assistant\tartan_data\kbtp\2020\10\10-22-20_audio
   Weather: C:\xcas-ga-comms-assistant\tartan_data\weather\BTP.csv


In [4]:
# ── Load ADS-B CSV ───────────────────────────────────────────────────────────
# The raw data has list-formatted strings for Time and Date columns
# e.g., Time = "[u'08', u'20', u'08.935']" — we'll parse these properly

df_raw = pd.read_csv(ADSB_PATH)

print("═" * 60)
print("ADS-B RAW DATA PROFILE")
print("═" * 60)
print(f"Shape          : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\nColumn names   : {list(df_raw.columns)}")
print(f"\nData types:\n{df_raw.dtypes}")
print(f"\nNull counts:\n{df_raw.isnull().sum()}")
print(f"\nFirst 3 rows:")
display(df_raw.head(3))

════════════════════════════════════════════════════════════
ADS-B RAW DATA PROFILE
════════════════════════════════════════════════════════════
Shape          : 233,868 rows × 12 columns

Column names   : ['ID', 'Time', 'Date', 'Altitude', 'Speed', 'Heading', 'Lat', 'Lon', 'Age', 'Range', 'Bearing', 'Tail']

Data types:
ID          float64
Time            str
Date            str
Altitude    float64
Speed       float64
Heading     float64
Lat         float64
Lon         float64
Age         float64
Range       float64
Bearing     float64
Tail            str
dtype: object

Null counts:
ID           722
Time         722
Date         722
Altitude     722
Speed       1923
Heading     1924
Lat          723
Lon          723
Age          723
Range        723
Bearing      723
Tail        1547
dtype: int64

First 3 rows:


,ID,Time,Date,Altitude,Speed,Heading,Lat,Lon,Age,Range,Bearing,Tail
0,11337771.0000,"[u'08', u'20', u'08.935']","[u'2020', u'10', u'22']",9400.0000,263.0000,268.0000,40.5121,-79.6006,2.7063,41.7557,134.8587,FDX1986
1,11337771.0000,"[u'08', u'20', u'11.646']","[u'2020', u'10', u'22']",9400.0000,263.0000,268.0000,40.5121,-79.6006,0.7098,41.7557,134.8587,FDX1986
2,11337771.0000,"[u'08', u'20', u'11.866']","[u'2020', u'10', u'22']",9400.0000,263.0000,268.0000,40.5121,-79.6006,1.7115,41.7557,134.8587,FDX1986


In [5]:
def parse_adsb_time(time_val, date_val):
    """
    Convert ADS-B list-formatted time/date strings to a proper datetime.
    
    Input format examples:
      time_val : "[u'08', u'20', u'08.935']"  OR  "['08', '20', '08.935']"
      date_val : "[u'2020', u'10', '22']"
    
    Returns: pandas Timestamp or NaT if parsing fails
    """
    try:
        # Extract numbers using regex — finds all digit sequences including decimals
        # re.findall returns a list of all matches
        t_parts = re.findall(r"[\d.]+", str(time_val))
        d_parts = re.findall(r"\d+", str(date_val))
        
        if len(t_parts) < 3 or len(d_parts) < 3:
            return pd.NaT
        
        hh, mm = int(t_parts[0]), int(t_parts[1])
        ss_float = float(t_parts[2])
        ss = int(ss_float)
        us = int((ss_float - ss) * 1_000_000)   # microseconds
        
        year, month, day = int(d_parts[0]), int(d_parts[1]), int(d_parts[2])
        
        return pd.Timestamp(year=year, month=month, day=day,
                            hour=hh, minute=mm, second=ss, microsecond=us)
    except Exception as e:
        return pd.NaT


# Apply parser to every row
# .apply() runs a function on every row — this is the pandas way to vectorise custom logic
df_raw["timestamp"] = df_raw.apply(
    lambda row: parse_adsb_time(row["Time"], row["Date"]),
    axis=1   # axis=1 means row-wise (axis=0 is column-wise)
)

# Report parsing success rate
n_total    = len(df_raw)
n_parsed   = df_raw["timestamp"].notna().sum()
n_failed   = n_total - n_parsed

print(f"Timestamp parsing results:")
print(f"  Total rows : {n_total:,}")
print(f"  Parsed OK  : {n_parsed:,} ({100*n_parsed/n_total:.1f}%)")
print(f"  Failed     : {n_failed:,} ({100*n_failed/n_total:.1f}%)")
print(f"\nTime range : {df_raw['timestamp'].min()} → {df_raw['timestamp'].max()}")
print(f"\nSample parsed timestamps:")
display(df_raw[["Time", "Date", "timestamp"]].head(5))

Timestamp parsing results:
  Total rows : 233,868
  Parsed OK  : 233,146 (99.7%)
  Failed     : 722 (0.3%)

Time range : 2020-10-22 08:20:08.935000 → 2020-10-23 02:59:56.012999

Sample parsed timestamps:


,Time,Date,timestamp
0,"[u'08', u'20', u'08.935']","[u'2020', u'10', u'22']",2020-10-22 08:20:08.935000
1,"[u'08', u'20', u'11.646']","[u'2020', u'10', u'22']",2020-10-22 08:20:11.646000
2,"[u'08', u'20', u'11.866']","[u'2020', u'10', u'22']",2020-10-22 08:20:11.865999
3,"[u'08', u'20', u'11.866']","[u'2020', u'10', u'22']",2020-10-22 08:20:11.865999
4,"[u'08', u'20', u'11.866']","[u'2020', u'10', u'22']",2020-10-22 08:20:11.865999


In [6]:
def parse_audio_txt(txt_path: Path) -> dict | None:
    """
    Parse a TartanAviation .txt sidecar file and extract:
      - start_time  (datetime)
      - end_time    (datetime)
      - duration_s  (float, seconds)
      - wav_path    (Path to corresponding .wav file)
      - has_metar   (bool)
    
    Expected format:
      Start Time: 2020-10-01 06:39:50.005130
      No metar data   ← optional line
      End Time:   2020-10-01 06:47:12.590370
      Total Time: 0:07:22.585255
    """
    try:
        text = txt_path.read_text(encoding="utf-8", errors="ignore")
        
        # Regex patterns — note the named groups (?P<name>...)
        # This makes matches accessible by name: m.group("dt")
        start_match = re.search(
            r"Start Time:\s*(?P<dt>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}[\.\d]*)",
            text
        )
        end_match = re.search(
            r"End Time:\s*(?P<dt>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}[\.\d]*)",
            text
        )
        
        if not start_match or not end_match:
            print(f"  ⚠ Could not parse times in: {txt_path.name}")
            return None
        
        start_dt = pd.Timestamp(start_match.group("dt"))
        end_dt   = pd.Timestamp(end_match.group("dt"))
        duration = (end_dt - start_dt).total_seconds()
        
        # Corresponding .wav file has the same stem
        wav_path = txt_path.with_suffix(".wav")
        
        return {
            "txt_file"   : txt_path.name,
            "wav_file"   : wav_path.name,
            "wav_path"   : str(wav_path),
            "start_time" : start_dt,
            "end_time"   : end_dt,
            "duration_s" : duration,
            "has_metar"  : "no metar" not in text.lower(),
            "txt_path"   : str(txt_path),
        }
    except Exception as e:
        print(f"  ✗ Error parsing {txt_path.name}: {e}")
        return None


# ── Scan all .txt files in the audio directory ───────────────────────────────
txt_files = sorted(AUDIO_DIR.glob("*.txt"))
print(f"Found {len(txt_files)} .txt files in {AUDIO_DIR.name}")

records = []
for txt_path in txt_files:
    rec = parse_audio_txt(txt_path)
    if rec:
        records.append(rec)

df_audio = pd.DataFrame(records)

print(f"\n✅ Audio manifest: {len(df_audio)} segments parsed")
print(f"\nTime coverage: {df_audio['start_time'].min()} → {df_audio['end_time'].max()}")
print(f"Total audio   : {df_audio['duration_s'].sum()/3600:.2f} hours")
print(f"With METAR    : {df_audio['has_metar'].sum()} segments")
print(f"\nSample:")
display(df_audio.head(5))

Found 52 .txt files in 10-22-20_audio

✅ Audio manifest: 52 segments parsed

Time coverage: 2020-10-22 11:53:33.138336 → 2020-10-23 02:38:55.341183
Total audio   : 10.33 hours
With METAR    : 0 segments

Sample:


,txt_file,wav_file,wav_path,start_time,end_time,duration_s,has_metar,txt_path
0,10.txt,10.wav,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...,2020-10-22 13:48:27.691478,2020-10-22 13:51:18.216862,170.5254,False,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...
1,12.txt,12.wav,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...,2020-10-22 14:07:39.231442,2020-10-22 14:22:39.740926,900.5095,False,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...
2,13.txt,13.wav,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...,2020-10-22 14:22:46.435089,2020-10-22 14:37:46.641410,900.2063,False,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...
3,14.txt,14.wav,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...,2020-10-22 14:37:53.105339,2020-10-22 14:52:53.418489,900.3131,False,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...
4,15.txt,15.wav,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...,2020-10-22 14:52:59.762546,2020-10-22 15:08:00.582346,900.8198,False,C:\xcas-ga-comms-assistant\tartan_data\kbtp\20...


In [ ]:
# Keep only rows that parsed successfully AND are from 2020-10-22
df_adsb = df_raw[
    df_raw["timestamp"].notna() &
    (df_raw["timestamp"].dt.date == pd.Timestamp("2020-10-22").date())
].copy()

# Clean up numeric columns — coerce errors to NaN (safe conversion)
for col in ["Altitude", "Speed", "Heading", "Lat", "Lon", "Range", "Bearing"]:
    df_adsb[col] = pd.to_numeric(df_adsb[col], errors="coerce")

# Sort chronologically
df_adsb = df_adsb.sort_values("timestamp").reset_index(drop=True)

print(f"ADS-B rows for 2020-10-22 : {len(df_adsb):,}")
print(f"Unique aircraft (Tail)    : {df_adsb['Tail'].nunique()}")
print(f"Time window               : {df_adsb['timestamp'].min().time()} → {df_adsb['timestamp'].max().time()} EDT")
print(f"\nAltitude range   : {df_adsb['Altitude'].min():.0f} – {df_adsb['Altitude'].max():.0f} ft MSL")
print(f"Speed range      : {df_adsb['Speed'].min():.0f} – {df_adsb['Speed'].max():.0f} kts")
print(f"Range from KBTP  : {df_adsb['Range'].min():.1f} – {df_adsb['Range'].max():.1f} km")
print(f"\nTop 10 aircraft by observation count:")
display(df_adsb['Tail'].value_counts().head(10).reset_index())

ADS-B rows for 2020-10-22 : 219,622
Unique aircraft (Tail)    : 257
Time window               : 08:20:08.935000 → 23:59:59.194000 UTC

Altitude range   : 0 – 17975 ft MSL
Speed range      : 0 – 432 kts
Range from KBTP  : 0.0 – 113.4 km

Top 10 aircraft by observation count:


,Tail,count
0,N819CM,13504
1,N53226,10914
2,N92141,9219
3,N13337,8784
4,N24TS,8105
5,N6683S,7652
6,N41548,7037
7,N43488,5921
8,N1452U,5309
9,N600HJ,5205


In [8]:
# ── Unified manifest: join ADS-B time window with audio segments ─────────────
# For each audio segment, we'll later find all ADS-B pings that fall within
# [start_time, end_time] — this is the temporal join we'll use in Project 1

manifest_records = []

for _, audio_row in df_audio.iterrows():
    # Find ADS-B pings during this audio segment
    mask = (
        (df_adsb["timestamp"] >= audio_row["start_time"]) &
        (df_adsb["timestamp"] <= audio_row["end_time"])
    )
    adsb_in_window = df_adsb[mask]
    
    manifest_records.append({
        "wav_file"         : audio_row["wav_file"],
        "wav_path"         : audio_row["wav_path"],
        "start_time"       : audio_row["start_time"],
        "end_time"         : audio_row["end_time"],
        "duration_s"       : audio_row["duration_s"],
        "has_metar"        : audio_row["has_metar"],
        "n_adsb_pings"     : len(adsb_in_window),
        "n_aircraft"       : adsb_in_window["Tail"].nunique() if len(adsb_in_window) > 0 else 0,
        "aircraft_in_window": sorted(adsb_in_window["Tail"].dropna().unique().tolist()),
    })

df_manifest = pd.DataFrame(manifest_records)

# Save to interim
manifest_path = INTERIM_DIR / "manifest_2020-10-22.csv"
df_manifest.to_csv(manifest_path, index=False)

# Save clean ADS-B
adsb_clean_path = INTERIM_DIR / "adsb_2020-10-22_clean.csv"
df_adsb.to_csv(adsb_clean_path, index=False)

print(f"✅ Manifest saved   → {manifest_path}")
print(f"✅ ADS-B CSV saved  → {adsb_clean_path}")
print(f"\nManifest summary:")
print(f"  Audio segments total   : {len(df_manifest)}")
print(f"  Segments with ADS-B    : {(df_manifest['n_adsb_pings'] > 0).sum()}")
print(f"  Avg aircraft per window: {df_manifest['n_aircraft'].mean():.1f}")
print(f"\nSample manifest rows:")
display(df_manifest[["wav_file","start_time","duration_s","n_adsb_pings","n_aircraft"]].head(10))

✅ Manifest saved   → C:\xcas-ga-comms-assistant\data\interim\manifest_2020-10-22.csv
✅ ADS-B CSV saved  → C:\xcas-ga-comms-assistant\data\interim\adsb_2020-10-22_clean.csv

Manifest summary:
  Audio segments total   : 52
  Segments with ADS-B    : 46
  Avg aircraft per window: 12.0

Sample manifest rows:


,wav_file,start_time,duration_s,n_adsb_pings,n_aircraft
0,10.wav,2020-10-22 13:48:27.691478,170.5254,237,3
1,12.wav,2020-10-22 14:07:39.231442,900.5095,1755,5
2,13.wav,2020-10-22 14:22:46.435089,900.2063,3090,11
3,14.wav,2020-10-22 14:37:53.105339,900.3131,6529,16
4,15.wav,2020-10-22 14:52:59.762546,900.8198,5415,14
5,16.wav,2020-10-22 15:08:07.120001,900.9394,5725,17
6,17.wav,2020-10-22 15:23:14.663441,376.2650,1555,10
7,18.wav,2020-10-22 15:39:13.891967,900.5252,6589,22
8,19.wav,2020-10-22 15:54:20.945662,900.8449,9046,22
9,20.wav,2020-10-22 16:09:28.373036,562.7232,3573,13


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import math

# Give each unique tail a consistent colour
unique_tails = df_adsb["Tail"].dropna().unique()
colour_palette = px.colors.qualitative.Vivid
tail_colours = {tail: colour_palette[i % len(colour_palette)]
                for i, tail in enumerate(sorted(unique_tails))}

fig = go.Figure()

# ── Plot each aircraft's track using new Scattermap API ──────────────────────
for tail, group in df_adsb.groupby("Tail"):
    group = group.sort_values("timestamp")
    colour = tail_colours.get(tail, "white")

    # Format timestamp nicely for hover
    hover_times = group["timestamp"].dt.strftime("%H:%M:%S UTC").tolist()

    fig.add_trace(go.Scattermap(
        lat=group["Lat"].tolist(),
        lon=group["Lon"].tolist(),
        mode="lines+markers",
        name=str(tail),
        line=dict(width=1, color=colour),
        marker=dict(size=4, color=colour),
        hovertemplate=(
            f"<b>{tail}</b><br>"
            "Time: %{customdata[0]}<br>"
            "Alt: %{customdata[1]:.0f} ft<br>"
            "Speed: %{customdata[2]:.0f} kts<br>"
            "Hdg: %{customdata[3]:.0f}°<br>"
            "Range: %{customdata[4]:.1f} km<extra></extra>"
        ),
        customdata=list(zip(
            hover_times,
            group["Altitude"].tolist(),
            group["Speed"].tolist(),
            group["Heading"].tolist(),
            group["Range"].tolist(),
        )),
    ))

# ── Airport marker ────────────────────────────────────────────────────────────
fig.add_trace(go.Scattermap(
    lat=[KBTP_LAT],
    lon=[KBTP_LON],
    mode="markers+text",
    marker=dict(size=16, color="yellow"),
    text=["KBTP"],
    textposition="top right",
    textfont=dict(color="yellow", size=13),
    name="KBTP Airport",
    hovertemplate="<b>KBTP Butler County Regional</b><br>"
                  "Lat: 40.7769°N  Lon: 79.9697°W<extra></extra>",
))

# ── Range rings (3 NM, 5 NM, 10 NM) ─────────────────────────────────────────
def make_ring(lat_center, lon_center, radius_nm, n_points=180):
    """Generate lat/lon circle points on Earth's surface using spherical trig."""
    radius_km = radius_nm * 1.852   # 1 NM = 1.852 km exactly (ICAO standard)
    R = 6371.0                       # Earth mean radius in km
    lats, lons = [], []
    for i in range(n_points + 1):
        bearing = math.radians(i * 360 / n_points)
        d = radius_km / R            # angular distance in radians
        lat1 = math.radians(lat_center)
        lon1 = math.radians(lon_center)
        lat2 = math.asin(
            math.sin(lat1) * math.cos(d) +
            math.cos(lat1) * math.sin(d) * math.cos(bearing)
        )
        lon2 = lon1 + math.atan2(
            math.sin(bearing) * math.sin(d) * math.cos(lat1),
            math.cos(d) - math.sin(lat1) * math.sin(lat2)
        )
        lats.append(math.degrees(lat2))
        lons.append(math.degrees(lon2))
    return lats, lons

ring_styles = [
    (10, "10 NM — Callout Zone",  "rgba(255,100,100,0.7)"),
    (5,  "5 NM — Callout Zone",   "rgba(255,200,50,0.7)"),
    (3,  "3 NM — Final Approach", "rgba(100,255,100,0.7)"),
]

for radius_nm, label, colour in ring_styles:
    rlats, rlons = make_ring(KBTP_LAT, KBTP_LON, radius_nm)
    fig.add_trace(go.Scattermap(
        lat=rlats,
        lon=rlons,
        mode="lines",
        line=dict(color=colour, width=1.5),   # dash not supported → use opacity
        name=label,
        hoverinfo="skip",
    ))

# ── Layout using new map (not mapbox) key ────────────────────────────────────
fig.update_layout(
    map=dict(                        # ← "map" not "mapbox" in new API
        style="carto-darkmatter",
        center=dict(lat=KBTP_LAT, lon=KBTP_LON),
        zoom=9,
    ),
    title=dict(
        text="KBTP Aircraft Tracks — 2020-10-22  |  Coloured by Tail Number",
        font=dict(color="white", size=16),
    ),
    paper_bgcolor="#0A1628",
    legend=dict(
        bgcolor="#1A3A5C",
        font=dict(color="white"),
        title_font_color="cyan",
        title_text="Aircraft / Zones",
    ),
    height=650,
    margin=dict(l=0, r=0, t=50, b=0),
)

fig.show()

# Save interactive HTML for GitHub
out_html = PROJECT_ROOT / "outputs" / "kbtp_traffic_map_2020-10-22.html"
fig.write_html(str(out_html))
print(f"✅ Interactive map saved → {out_html}")
print(f"   Aircraft tracks plotted : {df_adsb['Tail'].nunique()}")
print(f"   Total ADS-B pings       : {len(df_adsb):,}")